# F1 Session Results Metric View

Creates a Unity Catalog metric view over `formula1_incr.gold.fact_session_results` joined to `dim_races` (on season, round) to compute Total Wins by race, season, driver, and constructor.

In [0]:
%sql
CREATE OR REPLACE VIEW formula1_incr.gold.session_results_metrics
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: formula1_incr.gold.fact_session_results
  comment: Session result metrics (wins) joined to race details
  joins:
    - name: races
      source: formula1_incr.gold.dim_races
      on: source.season = races.season AND source.round = races.round
      cardinality: many_to_one
  dimensions:
    - name: season
      expr: source.season
    - name: round
      expr: source.round
    - name: session_type
      expr: source.session_type
    - name: driver_id
      expr: source.driver_id
    - name: constructor_id
      expr: source.constructor_id
    - name: race_name
      expr: races.race_name
    - name: race_date
      expr: races.race_date
    - name: circuit_name
      expr: races.circuit_name
    - name: country
      expr: races.country
  measures:
    - name: total_wins
      expr: COUNT_IF(source.is_win)
      comment: Number of session results where the driver won (is_win = true)
      synonyms:
        - wins
        - victories
$$

In [0]:
%sql
SELECT
  season,
  driver_id,
  MEASURE(total_wins) AS total_wins
FROM formula1_incr.gold.session_results_metrics
GROUP BY ALL
HAVING total_wins > 0
ORDER BY total_wins DESC